# This notebook will establish the database connection and build tables to be used later in modeling

In [1]:
from sqlalchemy import create_engine, text
import pandas as pd
from Database_Connection import make_engine
import numpy as np

### Assign connection variables

In [2]:
USER = ""
PASSWORD = ""
HOST = ""
PORT = 
DB = ""

In [3]:
engine = make_engine(USER, PASSWORD, HOST, PORT, DB)


Connected to hockey_stats as teammate1 @ 172.31.29.186:5432


## Read player-GAME Data
Player-game features are sourced from mart.player_game_features_{season}_truth

In [4]:
SEASONS = ["20182019","20192020","20202021","20212022","20222023","20232024","20242025"]

In [5]:
def read_player_game_truth_all(engine, seasons):
    selects = []
    for s in seasons:
        selects.append(f"""
            SELECT
                season::text AS season,
                game_id,
                player_id,
                team_id,
                cf, ca,
                toi_total_sec,
                toi_es_sec,
                cf60, ca60, cf_percent,
                goals, assists, points,
                shots, hits, blocked,
                takeaways, giveaways,
                faceoff_wins, faceoff_taken,
                penalties_taken
            FROM mart.player_game_features_{s}_truth
        """)
    q = " UNION ALL ".join(selects)
    return pd.read_sql(text(q), engine)


In [6]:

player_game = read_player_game_truth_all(engine, SEASONS)
player_game.shape, player_game.head()

((304430, 22),
      season     game_id  player_id  team_id  cf  ca  toi_total_sec  \
 0  20182019  2018020001    8466139       10  19  17           1011   
 1  20182019  2018020001    8468493       10  18  16           1208   
 2  20182019  2018020001    8473507        8  15  21           1437   
 3  20182019  2018020001    8474038        8  16  12            883   
 4  20182019  2018020001    8474581       10  20  23           1349   
 
    toi_es_sec       cf60       ca60  ...  assists  points  shots  hits  \
 0         927  73.786408  66.019417  ...        0       0      1     0   
 1         984  65.853659  58.536585  ...        0       0      1     1   
 2        1105  48.868778  68.416290  ...        0       0      1     2   
 3         788  73.096447  54.822335  ...        1       1      2     3   
 4        1155  62.337662  71.688312  ...        0       0      0     2   
 
    blocked  takeaways  giveaways  faceoff_wins  faceoff_taken  penalties_taken  
 0        4          0 

### Build team-game even strength from player_game

In [8]:
TEAM_SUM_COLS = [
    "toi_es_sec", "toi_total_sec",
    "cf","ca",
    "goals","assists","points",
    "shots","hits","blocked",
    "takeaways","giveaways",
    "faceoff_wins","faceoff_taken",
    "penalties_taken"
]

In [9]:
def build_team_game_from_player_game(pg: pd.DataFrame) -> pd.DataFrame:
    tg = (
        pg.groupby(["season","game_id","team_id"], as_index=False)[TEAM_SUM_COLS]
          .sum()
    )

    # rates from aggregated sums (use ES TOI for CF/CA rates)
    tg["cf_percent"] = tg["cf"] / (tg["cf"] + tg["ca"]).replace({0: np.nan})
    tg["cf60"] = (tg["cf"] / tg["toi_es_sec"].replace({0: np.nan})) * 3600.0
    tg["ca60"] = (tg["ca"] / tg["toi_es_sec"].replace({0: np.nan})) * 3600.0

    # faceoff win%
    tg["fo_percent"] = tg["faceoff_wins"] / tg["faceoff_taken"].replace({0: np.nan})

    # clean inf/nan created by denom=0
    tg = tg.replace([np.inf, -np.inf], np.nan)

    return tg

In [10]:
team_game_feats = build_team_game_from_player_game(player_game)
team_game_feats.shape, team_game_feats.head()

((16938, 22),
      season     game_id  team_id  toi_es_sec  toi_total_sec   cf   ca  goals  \
 0  20182019  2018020001        8       14706          17828  280  277      1   
 1  20182019  2018020001       10       14706          17740  277  280      1   
 2  20182019  2018020002        6       14510          17528  205  240      0   
 3  20182019  2018020002       15       14510          17764  240  205      3   
 4  20182019  2018020003       20       12785          17963  225  185      2   
 
    assists  points  ...  blocked  takeaways  giveaways  faceoff_wins  \
 0        2       3  ...       14          5          8            17   
 1        2       3  ...       21          3         20             0   
 2        0       0  ...       10          4          3            32   
 3        6       9  ...       14         11          9             0   
 4        4       6  ...        5          1          4            28   
 
    faceoff_taken  penalties_taken  cf_percent       cf60 

### Read team-game labels to get goals_for/goals_against/win across all seasons

In [11]:
def read_team_labels_all(engine, seasons):
    selects = []
    for s in seasons:
        selects.append(f"""
            SELECT
                '{s}'::text AS season,
                game_id,
                team_id,
                goals AS goals_for,
                goals_against,
                CASE WHEN goals > goals_against THEN 1 ELSE 0 END AS win
            FROM derived.team_event_totals_games_{s}
        """)
    q = " UNION ALL ".join(selects)
    return pd.read_sql(text(q), engine)


In [12]:

labels_all = read_team_labels_all(engine, SEASONS)
labels_all["season"] = labels_all["season"].astype(str)
labels_all.shape, labels_all.head()

((16938, 6),
      season     game_id  team_id  goals_for  goals_against  win
 0  20182019  2018020157       14          1              7    0
 1  20182019  2018020397        6          2              4    0
 2  20182019  2018020182       15          4              6    0
 3  20182019  2018020024       19          4              5    0
 4  20182019  2018020831       19          1              0    1)

### Join features + labels 

In [14]:
team_game = team_game_feats.merge(
    labels_all,
    on=["season","game_id","team_id"],
    how="inner"
)

team_game.shape, team_game.head()

((16938, 25),
      season     game_id  team_id  toi_es_sec  toi_total_sec   cf   ca  goals  \
 0  20182019  2018020001        8       14706          17828  280  277      1   
 1  20182019  2018020001       10       14706          17740  277  280      1   
 2  20182019  2018020002        6       14510          17528  205  240      0   
 3  20182019  2018020002       15       14510          17764  240  205      3   
 4  20182019  2018020003       20       12785          17963  225  185      2   
 
    assists  points  ...  faceoff_wins  faceoff_taken  penalties_taken  \
 0        2       3  ...            17             17                3   
 1        2       3  ...             0             30                3   
 2        0       0  ...            32             32                7   
 3        6       9  ...             0             16                4   
 4        4       6  ...            28             28                2   
 
    cf_percent       cf60       ca60  fo_percent  go

### Add opponent features

In [15]:
def add_opponent_features(df: pd.DataFrame) -> pd.DataFrame:
    opp = df.rename(columns={
        "team_id": "opp_team_id",
        "cf_percent": "opp_cf_percent",
        "cf60": "opp_cf60",
        "ca60": "opp_ca60",
        "fo_percent": "opp_fo_percent",
    })

    out = df.merge(
        opp[["season","game_id","opp_team_id","opp_cf_percent","opp_cf60","opp_ca60","opp_fo_percent"]],
        on=["season","game_id"],
        how="left"
    )

    out = out[out["team_id"] != out["opp_team_id"]].copy()

    out["diff_cf_percent"] = out["cf_percent"] - out["opp_cf_percent"]
    out["diff_cf60"] = out["cf60"] - out["opp_cf60"]
    out["diff_ca60"] = out["ca60"] - out["opp_ca60"]
    out["diff_fo_percent"] = out["fo_percent"] - out["opp_fo_percent"]

    return out

In [16]:
team_game = add_opponent_features(team_game)
team_game.shape

(16938, 34)

### Export artifacts

In [18]:
import os


OUTDIR = "data"
os.makedirs(OUTDIR, exist_ok=True)

In [19]:
player_path = os.path.join(OUTDIR, "player_game_truth_2018_2025.parquet")
team_path = os.path.join(OUTDIR, "team_game_truth_2018_2025.parquet")

player_game.to_parquet(player_path, index=False)
team_game.to_parquet(team_path, index=False)

print("Saved:", player_path)
print("Saved:", team_path)

Saved: data/player_game_truth_2018_2025.parquet
Saved: data/team_game_truth_2018_2025.parquet


### Export Archetypes

### Original Archetypes table

In [21]:
arctyps = pd.read_sql(
    text("""
        SELECT season::text AS season,
               player_id,
               cluster
        FROM mart.player_season_clusters_modern_truth
        WHERE season >= 20182019
    """),
    engine
)

arctyps.to_parquet("data/player_season_arctyps_modern_2018_2025.parquet", index=False)

In [23]:
print(arctyps.shape)
arctyps.head()

(6495, 3)


,season,player_id,cluster
0,20212022,8465009,2
1,20212022,8466138,1
2,20212022,8468503,2
3,20212022,8469455,0
4,20212022,8470281,2


### Archetypes with position labels (F/D)

In [26]:
arctyps_fd = pd.read_sql(
    text("""
        SELECT
            season::text AS season,
            player_id,
            pos_group,
            cluster
        FROM mart.v_player_season_archetypes_modern_regulars
        WHERE season BETWEEN 20182019 AND 20242025
    """),
    engine
)

arctyps_fd.to_parquet(
    "data/player_season_arctyps_fd_2018_2025.parquet",
    index=False
)

print(arctyps_fd.shape)
arctyps_fd.head()

(4753, 4)


,season,player_id,pos_group,cluster
0,20182019,8464989,F,1
1,20182019,8466138,F,0
2,20182019,8466139,F,1
3,20182019,8468508,F,0
4,20182019,8469454,F,1


In [45]:
base_labels = pd.read_sql(
    text("SELECT cluster, label FROM mart.player_cluster_labels_modern"),
    engine
)

cluster_labels = pd.concat([
    base_labels.assign(pos_group="F", description=""),
    base_labels.assign(pos_group="D", description="")
], ignore_index=True)[["pos_group", "cluster", "label", "description"]].sort_values(["pos_group","cluster"])

cluster_labels.to_parquet("data/cluster_labels_from_modern.parquet", index=False)

In [46]:
cluster_labels

,pos_group,cluster,label,description
3,D,0,two-way / possession-leaning,
4,D,1,checking / physical depth,
5,D,2,scoring / play-driving,
0,F,0,two-way / possession-leaning,
1,F,1,checking / physical depth,
2,F,2,scoring / play-driving,
